# Trading ML Pipeline — RunPod Jupyter (cascade_v2)
## Arsitektur 2-Model Cascade: LGBM (entry signal) → LSTM (confirmation vote)
## Workspace: /workspace/Riset_pemodelan

In [ ]:
# Clone repo jika belum ada, pull jika sudah ada
import os
if not os.path.exists("/workspace/Riset_pemodelan"):
    !git clone https://github.com/heathclif-cyber/Riset_pemodelan.git /workspace/Riset_pemodelan
else:
    %cd /workspace/Riset_pemodelan
    !git pull origin main

In [ ]:
%cd /workspace/Riset_pemodelan

In [ ]:
!pip install -q \
    lightgbm \
    torch \
    scikit-learn \
    shap \
    pyarrow \
    joblib \
    pandas \
    numpy \
    matplotlib \
    seaborn \
    ipywidgets

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {mem_gb:.1f} GB")
else:
    print("⚠️  GPU tidak tersedia — LSTM training akan lambat")

In [ ]:
%cd /workspace/Riset_pemodelan

In [ ]:
# Fetch semua 18 koin (training + new)
# Estimasi waktu: 30-60 menit pertama kali
# Gunakan --reset hanya jika ingin re-fetch dari awal
!python pipeline/01_fetch.py --all

In [ ]:
# Verifikasi data hasil fetch
import os
from pathlib import Path

raw_dir = Path("data/raw/klines")
if raw_dir.exists():
    coins = [d.name for d in raw_dir.iterdir() if d.is_dir()]
    print(f"Koin tersedia: {len(coins)} — {coins[:5]}...")
else:
    print("data/raw/klines belum ada — jalankan cell fetch terlebih dahulu")

In [ ]:
# Verifikasi data hasil fetch — semua koin
import os
from pathlib import Path

raw_dir = Path("data/raw/klines")
files = list(os.walk(raw_dir))
for root, dirs, fs in files[:8]:
    if fs:
        print(f"{Path(root).name}: {len(fs)} files")
print(f"... total {sum(len(fs) for _, _, fs in files if fs)} files")

In [ ]:
!python pipeline/02_clean.py --all

In [ ]:
# Fase 03 [OPSIONAL]: Grid Search Parameter Swing Labeling
# Mencari kombinasi optimal: SWING_H4_LOOKBACK, SWING_LABEL_MIN_RR, SWING_LABEL_MIN_TP, SWING_LABEL_MAX_SL
# Output: rekomendasi parameter — update config.py sebelum lanjut ke engineer
# Estimasi waktu: 10-20 menit
#
# Skip cell ini jika parameter swing sudah dikonfigurasi.
!python pipeline/03_analyze_swing.py --all --top 15

In [ ]:
# Verifikasi fitur baru (smart money v4) tersedia di feature engineering
!grep -n "calc_ofi_features\|calc_vwdp\|calc_vsa\|calc_trend_acceleration\|calc_volume_price_confirm\|calc_dist_from_recent_high" core/features.py

In [ ]:
# Fase 04: Feature Engineering + Swing Labeling
# Output: data/labeled/{SYMBOL}_features_v3.parquet
# Estimasi waktu: 15-30 menit
!python pipeline/04_engineer.py --all

In [ ]:
!python pipeline/analyze_min_hold.py --save-plot

In [ ]:
from IPython.display import Image, display
from pathlib import Path

# Tampilkan hasil analisis min_hold
img_path = Path("reports/min_hold_analysis.png")
if img_path.exists():
    display(Image(filename=str(img_path)))
else:
    print("min_hold_analysis.png belum ada — jalankan cell analyze_min_hold terlebih dahulu")

In [ ]:
# NOTE: H4 LGBM (04_train_lgbm_h4.py) sudah DIHAPUS dari arsitektur cascade_v2.
#
# Arsitektur sekarang: 2-Model Cascade
#   STEP 1  05_train_lgbm.py → LGBM entry signal (3-class: SHORT/FLAT/LONG)
#   STEP 2  06_train_lstm.py  → LSTM soft confirmation (tiered adjustment)
#
# Semua regime context (H4 trend, D1 alignment, volume confirmation, 
# correction detection) sudah embedded langsung sebagai 103 fitur di LGBM.
# H4 LGBM sebelumnya hanya AUC 0.55 (near-random) → dihapus.
#
# Logic cascade: pipeline/backtest_utils.py → hierarchical_predict()

print("cascade_v2 aktif. H4 LGBM dihapus. Lanjut ke LGBM training.")

In [ ]:
# Fase 05: Latih LGBM (Entry Signal Generator — Primary Model)
# 3-class: SHORT / FLAT / LONG
# Cost-sensitive class weights: SHORT=3x, FLAT=1.5x, LONG=3x
# Estimasi waktu: 15-30 menit
!python pipeline/05_train_lgbm.py --all

In [ ]:
# Verifikasi LGBM model sudah terlatih
from pathlib import Path
from config import MODEL_DIR

model_files = ["lgbm_baseline.pkl", "feature_cols_v2.json"]
for f in model_files:
    p = MODEL_DIR / f
    if p.exists():
        print(f"  ✅ {f}  ({p.stat().st_size / 1e6:.1f} MB)")
    else:
        print(f"  ❌ {f} — belum ada, jalankan 05_train_lgbm.py terlebih dahulu")

In [ ]:
# Fase 06: Latih LSTM (Confirmation Vote)
# LSTM memberikan soft adjustment terhadap confidence LGBM: agree=boost, opposite=penalty
# Input: sequence 16 bar H1 terakhir
# Estimasi waktu: 30-60 menit (GPU direkomendasikan)
!python pipeline/06_train_lstm.py --all

In [ ]:
# NOTE: Arsitektur cascade_v2 — 2-Model Cascade (LGBM + LSTM)
#
# Flow sinyal:
#   LGBM predict_proba() → confidence ≥ threshold 0.62?
#     → YES: lanjut ke LSTM soft adjustment
#     → NO:  FLAT (no trade)
#   LSTM adjustment (tiered):
#     agree (+0.05), neutral (-0.05), opposite (-0.08 × margin_multiplier)
#   Final confidence ≥ CONFIDENCE_THRESHOLD_ENTRY (0.70)?
#     → YES: sinyal diterbitkan (LONG/SHORT)
#     → NO:  FLAT
#
# Logic cascade: pipeline/backtest_utils.py → hierarchical_predict()
# TP/SL dinamis berbasis H4 Swing High/Low: core/evaluator.py → simulate_trades_swing()

print("cascade_v2 — 2 model. Tidak ada ensemble/H4 LGBM. Lanjut ke Fase 07.")

In [ ]:
# Verifikasi fungsi simulasi trade yang tersedia
!grep "def simulate_trades\|def calc_" core/evaluator.py

In [ ]:
import json
from pathlib import Path
from config import MODEL_DIR

registry = {
    "active": "cascade_v2",
    "architecture": "2-Model Cascade (LGBM → LSTM) with Dynamic Swing TP/SL",
    "confidence_threshold_entry": 0.70,
    "models": {
        "lgbm": {
            "role": "Entry Signal Generator (3-class: SHORT/FLAT/LONG)",
            "n_features": 103,
            "threshold_long": 0.62,
            "threshold_short": 0.62,
            "class_weights": {"SHORT": 3.0, "FLAT": 1.5, "LONG": 3.0},
        },
        "lstm": {
            "role": "Confirmation Vote (soft adjustment)",
            "seq_len": 16,
            "hidden": 128,
            "layers": 2,
            "adjust_mode": "tiered",
            "agree_boost": 0.05,
            "neutral_pen": 0.05,
            "opposite_pen": 0.08,
        }
    },
    "tp_sl": {
        "type": "dynamic_swing_h4",
        "min_rr": 1.2,
        "min_tp_atr": 1.2,
        "max_sl_atr": 3.0,
        "max_hold": 24,
    },
    "removed": ["h4_lgbm (AUC 0.55, near-random)", "stacked_ensemble (degradasi sinyal)"]
}

MODEL_DIR.mkdir(parents=True, exist_ok=True)
with open(MODEL_DIR / "model_registry.json", "w") as f:
    json.dump(registry, f, indent=2)

print("model_registry.json updated — cascade_v2")

In [ ]:
# Fase 07: Evaluasi Model + SHAP Analysis
# Output: models/runs/{run_id}/shap_h1_ranking.json
#         models/runs/{run_id}/cascade_metrics.json
#         models/runs/{run_id}/shap_importance.png
!python pipeline/07_evaluate.py

In [ ]:
# Fase 08: Walk-Forward Backtest (data training)
# Output: models/runs/{run_id}/backtest_results.json
#         models/runs/{run_id}/{SYMBOL}_trade_chart.svg
#         models/runs/{run_id}/backtest_summary_chart.svg
#         models/inference_config.json
!python pipeline/08_backtest.py --all

In [ ]:
# Fase 09: Hold-Out Backtest (OOS murni — data setelah periode training)
# Fetch data baru → Clean → Engineer → Backtest tanpa retrain
# Default: 2025-05-01 s/d 2026-04-01
# Output: models/runs/holdout_{run_id}/holdout_backtest_results.json
!python pipeline/09_holdout_backtest.py --all

In [ ]:
# Fase 10: Visualisasi Backtest + Swing Verification
# Output: reports/training/{SYMBOL}_backtest_visual.png
#         reports/training/{SYMBOL}_swing_verify.png (dengan --verify-swing)
!python pipeline/10_visualize.py --all --verify-swing

# Untuk data hold-out (OOS):
# !python pipeline/10_visualize.py --all --holdout --verify-swing